In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from dotenv import load_dotenv
load_dotenv('../.env')


True

In [2]:
import pandas as pd

df = pd.read_csv('../IIPC_data/IIPC_cleaned_text.csv')
df = df[df['cleaned_text'].notnull()].copy()
df['cleaned_text'] = df['cleaned_text'].astype(str)

In [3]:
from tqdm import tqdm
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

df['doc_id'] = df.index.astype(str)

CHUNK_SIZE = 512
OVERLAP = 50

def chunk_by_tokens(text, tokenizer, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap
    return chunks

chunked_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Token chunking"):
    text_chunks = chunk_by_tokens(row['cleaned_text'], tokenizer)
    for i, chunk in enumerate(text_chunks):
        new_row = row.to_dict()
        new_row['cleaned_text'] = chunk
        new_row['chunk_id'] = i
        chunked_rows.append(new_row)

df = pd.DataFrame(chunked_rows).reset_index(drop=True)

c:\Users\youss\AppData\Local\Programs\Python\Python39\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Token chunking:   0%|          | 0/615 [00:00<?, ?it/s]

Token chunking: 100%|██████████| 615/615 [00:17<00:00, 34.89it/s] 


In [4]:
import os
import requests
import time
import pickle
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv('../.env')

PKL_PATH = '../IIPC_data/embeddings_v3.pkl'

if os.path.exists(PKL_PATH):
    print(f"✅ Embeddings file already exists at {PKL_PATH}. Skipping generation.")
else:
    # Your Hugging Face Space URL (loaded from your .env file)
    SPACE_API_URL = os.getenv("VITE_EMBEDDING_API_URL")
    HF_TOKEN = os.getenv("HF_TOKEN")

    if not SPACE_API_URL:
        raise ValueError("⚠️ Please set VITE_EMBEDDING_API_URL in your .env file!")

    # Ensure the URL points to the /embed route
    if not SPACE_API_URL.endswith("/embed"):
        SPACE_API_URL = SPACE_API_URL.rstrip("/") + "/embed"

    print(f"📡 Using Private Hugging Face Space Embedding API: {SPACE_API_URL}")

    # Helper to get embeddings in batches from your own HuggingFace Space
    def get_embeddings_from_space(texts, batch_size=16):
        all_embeddings = []
        headers = {}
        if HF_TOKEN:
            headers["Authorization"] = f"Bearer {HF_TOKEN}"
            
        for i in tqdm(range(0, len(texts), batch_size), desc="Embedding via HF Space"):
            batch = texts[i:i+batch_size]
            for attempt in range(5):
                try:
                    # Send batch with authorization headers
                    response = requests.post(SPACE_API_URL, json={"text": batch}, headers=headers, timeout=60)
                    response.raise_for_status()
                    res_json = response.json()
                    all_embeddings.extend(res_json["embeddings"])
                    break
                except Exception as e:
                    print(f"\n⚠️ Error on batch {i} (attempt {attempt+1}/5): {e}. Retrying in 5s...")
                    time.sleep(5)
                    
        return all_embeddings

    df['combined_text'] = (
        "This item is titled '" + df['title'].fillna('') + 
        "' and was created by " + df['creator'].fillna('Unknown') + ". " +
        "It is about " + df['subject'].fillna('various topics') + ". " +
        "Description: " + df['description'].fillna('No description provided.') + " " +
        "Main content: " + df['cleaned_text'].fillna('') + " " +
        "It is classified as " + df['item_type'].fillna('unknown type') +
        ", dated " + df['date'].fillna('unknown date') + ". " +
        "You can find it at " + df['ark_url'].fillna('') + 
        " (source: " + df['source_url'].fillna('') + ")."
)

    # Generate embeddings using your HuggingFace Space endpoint
    embeddings = get_embeddings_from_space(df['combined_text'].tolist(), batch_size=16)
    embeddings = np.array(embeddings).astype('float32')

    os.makedirs('../IIPC_data', exist_ok=True)
    with open(PKL_PATH, 'wb') as f:
        pickle.dump({
            'embeddings': embeddings,
            'cleaned_texts': df['cleaned_text'].fillna('').tolist(),
            'doc_ids': df['doc_id'].tolist(),
            'chunk_id': df['chunk_id'].tolist(),
            'titles': df['title'].tolist(),
            'creators': df['creators'].tolist() if 'creators' in df.columns else df['creator'].tolist(),
            'dates': df['date'].tolist(),
            'ark_urls': df['ark_url'].tolist(),
            'subjects': df['subject'].tolist(),
            'descriptions': df['description'].tolist(),
            'item_types': df['item_type'].tolist(),
            'source_urls': df['source_url'].tolist()
        }, f)

    print("Embeddings and all metadata saved successfully via your HuggingFace Space API.")

✅ Embeddings file already exists at ../IIPC_data/embeddings_v3.pkl. Skipping generation.


In [5]:
import pickle
import numpy as np
import faiss

with open('../IIPC_data/embeddings_v3.pkl', 'rb') as f:
    data = pickle.load(f)

embeddings = np.array(data['embeddings']).astype('float32')
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [6]:
import os
import requests
import numpy as np
import faiss
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv('../.env')
HF_TOKEN = os.getenv("HF_TOKEN")
SPACE_API_URL = os.getenv("VITE_EMBEDDING_API_URL")
if SPACE_API_URL and not SPACE_API_URL.endswith("/embed"):
    SPACE_API_URL = SPACE_API_URL.rstrip("/") + "/embed"

# Helper to get query embedding from your private Hugging Face Space
def get_query_embedding(query_text):
    headers = {}
    if HF_TOKEN:
        headers["Authorization"] = f"Bearer {HF_TOKEN}"
    
    response = requests.post(SPACE_API_URL, json={"text": query_text}, headers=headers, timeout=30)
    response.raise_for_status()
    # Space returns a single vector inside {"embedding": [...]}
    return response.json()["embedding"]

def mmr(query_emb, candidate_embs, lambda_param=0.5, top_k=10):
    """Maximal Marginal Relevance (MMR) for diverse, non-repetitive retrieval."""
    if len(candidate_embs) == 0:
        return []
        
    selected = []
    candidates = list(range(len(candidate_embs)))
    
    # Precompute similarities to the query
    sim_to_query = np.dot(candidate_embs, query_emb)
    
    # Keep track of similarity of candidates to the selected set
    max_sim_to_selected = np.zeros(len(candidate_embs))
    
    while len(selected) < top_k and candidates:
        mmr_scores = lambda_param * sim_to_query[candidates] - (1 - lambda_param) * max_sim_to_selected[candidates]
        best_idx = candidates[np.argmax(mmr_scores)]
        
        selected.append(best_idx)
        candidates.remove(best_idx)
        
        # Update max similarity array with the newly selected embedding
        if candidates:
            sims = np.dot(candidate_embs[candidates], candidate_embs[best_idx])
            max_sim_to_selected[candidates] = np.maximum(max_sim_to_selected[candidates], sims)
            
    return selected

def retrieve_top_k(query, k_chunks=30, k_docs=3, k_final=10):
    """Search FAISS and return top diverse slides matching the query."""
    # Get embedding from Hugging Face Space
    q_emb = get_query_embedding(query)
    q_emb = np.array(q_emb).astype('float32').reshape(1, -1)
    faiss.normalize_L2(q_emb)

    # Search FAISS index
    distances, indices = index.search(q_emb, k_chunks)
    
    retrieved_chunks = []
    for idx, dist in zip(indices[0], distances[0]):
        retrieved_chunks.append({
            'idx': idx,
            'doc_id': data['doc_ids'][idx],
            'chunk_id': data['chunk_ids'][idx],
            'title': data['titles'][idx],
            'creator': data['creators'][idx],
            'date': data['dates'][idx],
            'ark_url': data['ark_urls'][idx] if 'ark_urls' in data else '',
            'cleaned_text': data['cleaned_texts'][idx],
            'score': float(dist),
            'subject': data['subjects'][idx] if 'subjects' in data else '',
            'description': data['descriptions'][idx] if 'descriptions' in data else '',
            'item_type': data['item_types'][idx] if 'item_types' in data else '',
            'source_url': data['source_urls'][idx] if 'source_urls' in data else ''
        })

    # Group candidate chunks by document
    doc_groups = defaultdict(list)
    for r in retrieved_chunks:
        doc_groups[r['doc_id']].append(r)

    # Filter to top k_docs (presentations) with highest single match score
    top_docs = sorted(doc_groups.items(), key=lambda x: max(c['score'] for c in x[1]), reverse=True)[:k_docs]

    # Gather all candidate chunks from these top presentations
    candidates = []
    for _, chunks in top_docs:
        candidates.extend(chunks)

    # Re-rank candidate chunks using MMR to extract the best diverse set
    candidate_vectors = np.array([embeddings[c['idx']] for c in candidates])
    mmr_selected_indices = mmr(q_emb.flatten(), candidate_vectors, lambda_param=0.5, top_k=k_final)

    return [candidates[i] for i in mmr_selected_indices]

In [7]:
import os
import google.generativeai as genai
from collections import defaultdict

# Initialize Gemini Client
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

MODEL_NAME = "gemini-3.1-flash-lite"
AiModel = genai.GenerativeModel(MODEL_NAME)

def generate_response(query, context_docs):
    """Generate response using the exact prompt instructions from app.py."""
    grouped_docs = defaultdict(list)
    doc_metadata = {}
    
    # Group clean chunks and store unique document metadata
    for doc in context_docs:
        doc_id = doc['doc_id']
        grouped_docs[doc_id].append(doc['cleaned_text'])
        if doc_id not in doc_metadata:
            doc_metadata[doc_id] = {
                'title': doc.get('title', 'Unknown Title'),
                'creator': doc.get('creator', 'Unknown Creator'),
                'date': doc.get('date', 'Unknown Date'),
                'ark_url': doc.get('ark_url', '')
            }

    # Build clean context block for prompt injection
    context_parts = []
    for doc_id, chunks in grouped_docs.items():
        meta = doc_metadata[doc_id]
        doc_content = "\n".join(chunks)
        context_parts.append(
            f"SOURCE ID: {doc_id}\n"
            f"TITLE: {meta['title']}\n"
            f"CREATOR/AFFILIATION: {meta['creator']}\n"
            f"DATE: {meta['date']}\n"
            f"URL: {meta['ark_url']}\n"
            f"CONTENT CHUNKS:\n{doc_content}\n----"
        )
    context = "\n\n".join(context_parts)

    # Exact Prompt structure from app.py
    prompt = f"""You are an IIPC digital preservation and web archiving assistant. Answer using ONLY the provided conference materials below.

QUERY HANDLING:
- Greetings/capability questions: Respond briefly, no citations needed
- Substantive questions: Use context strictly, cite sources

RULES:
1. Never use outside knowledge. If insufficient info: "Based on available IIPC materials, I don't have enough information to fully answer this."
2. In-text citations: "According to [Author]'s '[Title]' ([Year])..." — NO ARK URLs inline
3. End substantive answers with a "Sources Referenced:" section:
   - [Title] by [Author] ([Year]): [ARK URL]
   (Each source listed once only)
4. Use precise web archiving terminology (WARC, etc.)
5. Synthesize across documents; note how topics evolved over conference years
6. Plain text only — no markdown except in Sources section

Context:
{context}

Question: {query}

Answer:"""

    # Query Gemini API
    response = AiModel.generate_content(prompt)
    return response.text

In [8]:
query = "What have the National Library of Norway presented about in IIPC Conferences?"
results = retrieve_top_k(query)
answer = generate_response(query, results)
print(answer)

The National Library of Norway has presented on utilizing their extensive web archive collections to provide research infrastructure and support data-driven analysis.

Most recently, in 2025, Tønnessen and Birkenes presented on the transformation of the library's web news collection—which includes 1.5 million texts across 268 publication titles—into usable corpus data. This work builds upon over a decade of institutional experience in digital humanities through their dedicated Lab for Digital Humanities (DH-lab). Their presentation emphasized the use of specialized tools, such as warc2corpus, to facilitate computational research while navigating complex legal constraints, including copyright and GDPR requirements for personal data.

These efforts represent an evolution from early institutional archiving toward providing "Collections as Data." This approach enables researchers to perform distant reading and other computational methods on large-scale web datasets that are free from the O